# AirQ+
Insumos:
- Contaminante (O₃, SO₂, PM10 o PM2.5)
- Población ≥30 años
- Promedio de muertes (2015–2019) para población ≥30 años


In [ ]:
#---------------------------------
#Importar librerias
#---------------------------------
import pandas as pd
import numpy as np
import os

In [ ]:
# Riesgo relativo por incremento de 10 µg/m³
HR_base = 1.04
HR_low_base = 1.03
HR_high_base = 1.06

# Escenarios contrafactuales (µg/m³)
escenarios = {
    "OMS_15": {"tipo": "fijo", "valor":15},
    "IT4_20": {"tipo": "fijo", "valor": 20},
    "PA_30": {"tipo": "fijo", "valor": 30},
    "REDUCCION_4": {"tipo": "porcentaje", "valor": 0.04}
}

In [ ]:
#---------------------------------
#Estandarizar datos de AirQ+
#---------------------------------
def estandarizar_airq(df):
    df = df.copy()

    rename_map = {
        "MAYORES_30AÑOS": "pop30",
        "promedio_muertes_2015_2019_naturales_30_años": "muertes",
        "MODELADO_PM10_2022": "pm10"
    }

    df.rename(columns=rename_map, inplace=True)

    # Validación mínima
    required = ["SETU_CCNCT", "pop30", "muertes", "pm10"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    return df

In [ ]:
#---------------------------------
#Función para calcular AirQ+ para PM10
#---------------------------------
def calcular_airq_pm10(df, escenario, sufijo):
    df = df.copy()

    if escenario["tipo"] == "fijo":
        contrafactual = escenario["valor"]
    elif escenario["tipo"] == "porcentaje":
        contrafactual = df["pm10"] * (1 - escenario["valor"])
    else:
        raise ValueError(f"Tipo de escenario no soportado: {escenario['tipo']}")

    # ΔPM10
    df["delta_pm10"] = df["pm10"] - contrafactual
    df["delta_pm10"] = np.where(df["delta_pm10"] > 0, df["delta_pm10"], 0)

    factor = df["delta_pm10"] / 10

    # Hazard Ratio ajustado
    df[f"HR_{sufijo}"] = HR_base ** factor
    df[f"HR_low_{sufijo}"] = HR_low_base ** factor
    df[f"HR_high_{sufijo}"] = HR_high_base ** factor

    # PAF
    df[f"PAF_{sufijo}"] = (df[f"HR_{sufijo}"] - 1) / df[f"HR_{sufijo}"]
    df[f"PAF_low_{sufijo}"] = (df[f"HR_low_{sufijo}"] - 1) / df[f"HR_low_{sufijo}"]
    df[f"PAF_high_{sufijo}"] = (df[f"HR_high_{sufijo}"] - 1) / df[f"HR_high_{sufijo}"]

    # Muertes atribuibles
    df[f"muertes_atr_{sufijo}"] = df["muertes"] * df[f"PAF_{sufijo}"]
    df[f"muertes_atr_low_{sufijo}"] = df["muertes"] * df[f"PAF_low_{sufijo}"]
    df[f"muertes_atr_high_{sufijo}"] = df["muertes"] * df[f"PAF_high_{sufijo}"]

    return df, contrafactual


In [ ]:
#---------------------------------
#Función para ejecutar modelo para todos los escenarios
#---------------------------------
def ejecutar_modelo(df, zona_nombre):
    resultados = []

    for nombre, escenario in escenarios.items():
        df_res, contrafactual = calcular_airq_pm10(df, escenario, nombre)

        output_path = f"AIRQ+/AIRQ_PM10_{zona_nombre}_{nombre}.csv"
        os.makedirs("AIRQ+", exist_ok=True)
        df_res.to_csv(output_path, index=False)

        if np.isscalar(contrafactual):
            contrafactual_resumen = contrafactual
        else:
            contrafactual_resumen = contrafactual.mean()

        resumen = {
            "Zona": zona_nombre,
            "Escenario": nombre,
            "Contrafactual": contrafactual_resumen,
            "Muertes_atrib_total": df_res[f"muertes_atr_{nombre}"].sum(),
            "Muertes_atrib_low": df_res[f"muertes_atr_low_{nombre}"].sum(),
            "Muertes_atrib_high": df_res[f"muertes_atr_high_{nombre}"].sum(),
            "PAF_promedio": df_res[f"PAF_{nombre}"].mean()
        }

        resultados.append(resumen)

    return pd.DataFrame(resultados)


In [ ]:
#---------------------------------
#Ejecutar METRO y ZUMA
#---------------------------------
# Leer bases
df_metro_raw = pd.read_excel("METRO_AIRQ+.xlsx", dtype={"SETU_CCNCT": str})
df_zuma_raw = pd.read_excel("ZUMA_AIRQ+.xlsx", dtype={"SETU_CCNCT": str})

# Estandarizar
df_metro = estandarizar_airq(df_metro_raw)
df_zuma = estandarizar_airq(df_zuma_raw)

# Ejecutar modelo
resumen_metro = ejecutar_modelo(df_metro, "METRO")
resumen_zuma = ejecutar_modelo(df_zuma, "ZUMA")

# Unir resultados
resumen_total = pd.concat([resumen_metro, resumen_zuma], ignore_index=True)

# Guardar resumen global
os.makedirs("outputs_aggregated", exist_ok=True)
resumen_total.to_csv("outputs_aggregated/AIRQ_PM10_summary.csv", index=False)

resumen_total

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

plt.style.use("ggplot")
os.makedirs("AIRQ+", exist_ok=True)

colores = {
    "OMS_15": "#0b3c5d",
    "IT4_20": "#1d70a2",
    "PA_30": "#74a9cf",
    "REDUCCION_4": "#b7d4ea"
}

labels_dict = {
    "OMS_15": "OMS (15 µg/m³)",
    "IT4_20": "OMS IT-4 (20 µg/m³)",
    "PA_30": "Plan Aire (30 µg/m³)",
    "REDUCCION_4": "Reducción del 4%"
}

orden_escenarios = {
    "OMS_15": 0,
    "IT4_20": 1,
    "PA_30": 2,
    "REDUCCION_4": 3
}

zonas = resumen_total["Zona"].unique()

for zona in zonas:
    sub = resumen_total[resumen_total["Zona"] == zona].copy()
    sub["orden"] = sub["Escenario"].map(orden_escenarios)
    sub = sub.sort_values("orden")

    fig, ax = plt.subplots(figsize=(9, 6))

    x = np.arange(len(sub))
    width = 0.8
    colores_barras = [colores[val] for val in sub["Escenario"]]

    ax.bar(
        x,
        sub["Muertes_atrib_total"],
        width,
        color=colores_barras,
        alpha=0.95
    )

    for xi, low, high, central in zip(
        x,
        sub["Muertes_atrib_low"],
        sub["Muertes_atrib_high"],
        sub["Muertes_atrib_total"]
    ):
        ax.plot([xi, xi], [low, high], color="black", linewidth=1.2)
        ax.plot([xi - 0.08, xi + 0.08], [low, low], color="black", linewidth=1.2)
        ax.plot([xi - 0.08, xi + 0.08], [high, high], color="black", linewidth=1.2)

        ax.text(xi, central / 2, f"{central:.0f}", ha="center", va="center",
                color="white", fontsize=12, fontweight="bold")
        ax.text(xi, low - 25, f"{low:.0f}", ha="center", va="bottom",
                color="black", fontsize=10, fontweight="bold")
        ax.text(xi, high, f"{high:.0f}", ha="center", va="bottom",
                color="black", fontsize=10, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([labels_dict[val] for val in sub["Escenario"]], fontsize=12)
    ax.set_ylabel("Muertes prematuras prevenibles", fontsize=16, weight="bold")
    ax.set_xlabel("Escenario contrafactual de PM10", fontsize=16, weight="bold")
    ax.set_title(
        "Muertes prematuras prevenibles debido a niveles modelados de PM10",
        fontsize=14,
        weight="bold"
    )

    ax.grid(axis="y", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"AIRQ+/muertes_prevenibles_pm10_{zona}.png", dpi=300, bbox_inches="tight")
    plt.show()